In [1]:
import os
import sys

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = "/scratch/shareddata/dldata/huggingface-hub-cache/hub" # Load local models
os.environ["TRANSFORMERS_OFFLINE"] = "1" 
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = f"{os.environ["WRKDIR"]}/.cache/huggingface" # Cache in work directory

sys.path.append(
    os.path.abspath(os.path.join(os.getcwd(), "../.."))
)  # Add parent directory to path

from collections import defaultdict
import json
import pandas as pd
import transformers
from datasets import load_dataset
import random 

from utils.helpers import parse_output

from utils.constants import DEFAULT_AUGMENT_RESULT, CONCEPT_TO_CHAPTER_MAPPING, THEME_TO_TOPICS_MAPPING

from utils.prompts import (
    JUDGE_SYSTEM_PROMPT,
    JUDGE_TEMPLATE,
    AUGMENT_SYSTEM_PROMPT,
    AUGMENT_TEMPLATE,
)

In [2]:
AUGMENT_SYSTEM_PROMPT = """You are a system that rewrites programming exercises.

You will receive:
- a theme
- a topic
- a programming concept
- a programming exercise consisting of a problem description and an example solution.

Your task:
Modify the exercise so that it follows the provided theme and topic and 
the program code in a way that it utilizes the programming concept in a non-trivial
way. The modified exercise must keep the same style as the original.

CRITICAL OUTPUT RULES:
- You must output ONLY a valid JSON object.
- Do not include explanations, comments, markdown, or code fences.
- The output must be valid JSON that can be parsed with a standard JSON parser.
- All strings must be properly escaped.
- The "code" field must be a JSON string containing the solution code.

JSON schema:

{
  "augmentedProblemDescription": "string",
  "augmentedExampleSolution": {
    "code": "string"
  }
}

Before finishing, verify that the output is valid JSON and follows the schema exactly.
"""

AUGMENT_TEMPLATE = """Rewrite the following programming exercise.

Theme: $THEME$
Topic: $TOPIC$
Concept: $CONCEPT$

--- ORIGINAL PROBLEM DESCRIPTION ---
$TEXT$

--- ORIGINAL EXAMPLE SOLUTION ---
$CODE$

Return the modified exercise as JSON following the required schema.
"""

DEFAULT_DATA = r"../../../data/complete_dataset.csv"
DEFAULT_MODEL = "Qwen/Qwen2.5-14B-Instruct"

system_prompts = {
    "judge": JUDGE_SYSTEM_PROMPT,
    "augment": AUGMENT_SYSTEM_PROMPT
}

In [3]:
def make_prompt(row, task_type):
    match task_type:
        case "judge":
            return (
                JUDGE_TEMPLATE.replace("$THEME$", row["theme"])
                .replace("$TOPIC$", row["topic"])
                .replace("$CONCEPT$", row["concept"])
                .replace("$TEXT$", row["problemDescription"])
                .replace("$CODE$", row["exampleSolution"])
            )
        case "augment":
            new_theme = random.choice(list(THEME_TO_TOPICS_MAPPING.keys()))
            # Ensure topic is not the same
            new_topic = random.choice(
                list(filter(lambda x: x != row["topic"], THEME_TO_TOPICS_MAPPING.get(new_theme)))
            )
            
            concept = row["concept"]
            concept_chapter = CONCEPT_TO_CHAPTER_MAPPING.get(concept)
            num_chapters = len(set(CONCEPT_TO_CHAPTER_MAPPING.values()))
            next_chapter = random.randint(concept_chapter + 1, num_chapters)
            
            advanced_concept = random.choice(
                list(filter(
                    lambda v: v[1] == next_chapter , CONCEPT_TO_CHAPTER_MAPPING.items())
                ))[0]
            
            return (
                AUGMENT_TEMPLATE.replace("$THEME$", new_theme)
                .replace("$TOPIC$", new_topic)
                .replace("$CONCEPT$", advanced_concept)
                .replace("$TEXT$", row["problemDescription"])
                .replace("$CODE$", row["exampleSolution"])
            )
        case _:
            raise ValueError(f"Task type '{_}' not recognised as valid task type!")

In [4]:
task = "augment"
random.seed(42)
n_rows = 1
dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")

if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

dataset = dataset.map(lambda row: {"prompt": make_prompt(row, task)})

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [5]:
#print(dataset.to_pandas()["prompt"][0])

In [6]:
params = {
    "model": DEFAULT_MODEL,
    "device_map": 0,  # Force GPU
    "max_new_tokens": 1000,
    "temperature": 0.3,
}

pipeline = transformers.pipeline("text-generation", **params)
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


In [7]:
results = defaultdict(list)
for prompt in dataset["prompt"]:
    print("Prompting...")
    output = pipeline(
        [
            {"role": "system", "content": system_prompts.get(task)},
            {"role": "user", "content": prompt},
        ],
        batch_size=1,
        return_full_text=False,
    )

    text = output[0]["generated_text"]

    parsed = parse_output(text)

    if parsed.get("Error") is not None:
        for k, v in DEFAULT_AUGMENT_RESULT.items():
            results[k].append(v)
        continue # Skip to next prompt

    keys = ["augmentedProblemDescription", "augmentedExampleSolution"] # Map to named lists
    for k in keys:
        results[k].append(json.dumps(parsed.get(k, "PARSE ERROR")))

for k, v in results.items():  # Add named lists as columns
    dataset = dataset.add_column(k, v)

print("done")

Prompting...
Invalid \escape: line 4 column 84 (char 816)
done


In [8]:
for i, row in dataset.to_pandas().iterrows():
    print("Original theme: " + row["theme"])
    print("Original topic: " + row["topic"])
    print("Original concept: " + row["concept"])
    print()
    print(row["prompt"])
    print("")
    print(row["augmentedProblemDescription"])
    print("")
    print(row["augmentedExampleSolution"])
    print("------------------------------------------------")
    #print(row["theme"])

Original theme: literature
Original topic: Agatha Christie
Original concept: conditional statements

Rewrite the following programming exercise.

Theme: classical music
Topic: Henry Purcell
Concept: for loops

--- ORIGINAL PROBLEM DESCRIPTION ---
Agatha Christie, the famous novelist, has a rating scale for her novels. The ratings are represented as numbers and are accompanied by the following textual descriptions:
<table>
<tr>
<th>Rating</th>
<th>Description</th>
</tr>
<tr>
<th>5</th>
<th>Masterpiece</th>
</tr>
<tr>
<th>4</th>
<th>Excellent</th>
</tr>
<tr>
<th>3</th>
<th>Good</th>
</tr>
<tr>
<th>2</th>
<th>Fair</th>
</tr>
<tr>
<th>1</th>
<th>Below Average</th>
</tr>
</table>
Write a program that asks the user for a number and prints the textual description related to that number. If the user enters any other number, the program should print the message <code>Invalid rating!</code>.

Below is an example of the expected operation of the program.

<pre>
What rating?
<b>&lt; 3</b>
Good
</p